In [1]:
# Built on top of "TreeTrainer" object
# - keep a per query buffer, tracking trie of all previous rollout

# some toy rollouts on a specific Game of 24 queries in a TreeTrainer class 


## Per-query rollout buffer (trie of *all* previous rollouts)

The idea in two bullets:

- **Built on `TreeTrainer`** — reuse its `PrefixTrie` + the `use_global_tree` machinery, which already keys a persistent trie per prompt (`self._global_tries[pkey]`).
- **Per-query buffer** — every rollout ever seen for a query is folded into that query's trie. A prefix then inherits the *best reachable* advantage (`a_max`, OPA) over **all batches**, not just the current group.

Below: toy rollouts on one specific Game-of-24 query, first with a tiny `QueryBuffer`, then with the real `TreeTrainer._tree_token_advantages` core to show they're the same mechanism. No GPU / model needed.

In [2]:
import sys
from pathlib import Path

import torch

REPO = Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.tree_trainer import PrefixTrie, optimistic_prefix_advantages, TreeTrainer
from src.game24utils import verify_24

torch.manual_seed(0)
print("imports ok")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

objc[61921]: Class AVFFrameReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2c6700798) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x2e0e643a8). One of the two will be used. Which one is undefined.
objc[61921]: Class AVFAudioReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2c67007e8) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x2e0e643f8). One of the two will be used. Which one is undefined.


imports ok


In [3]:
# One specific Game-of-24 query we hammer with rollouts.
NUMBERS = (2, 4, 8, 10)

# Toy policy completions (final expressions) arriving over two successive
# training batches on the SAME query. Some hit 24, some don't. The prefixes are
# chosen to overlap ACROSS batches so the buffer has something to propagate.
BATCH0 = [
    "(10-4)*8/2",      # = 24  correct
    "(10-4)*(8-2)",    # = 36  wrong   (shares prefix "(10-4)*")
    "2*10+8-4",        # = 24  correct
    "2*10-8-4",        # =  8  wrong   (shares prefix "2*10")
]
BATCH1 = [
    "(10-4)*8-2",      # = 46  wrong   (shares prefix "(10-4)*8" with BATCH0's CORRECT one)
    "2*10+8+4",        # = 32  wrong
    "(10+2)*(8/4)",    # = 24  correct
    "8*4-10+2",        # = 24  correct
]

def reward(expr):
    return 1.0 if verify_24(NUMBERS, expr) else 0.0

for c in BATCH0 + BATCH1:
    print(f"{c:16s} -> 24? {bool(reward(c))}")

(10-4)*8/2       -> 24? True
(10-4)*(8-2)     -> 24? False
2*10+8-4         -> 24? True
2*10-8-4         -> 24? False
(10-4)*8-2       -> 24? False
2*10+8+4         -> 24? False
(10+2)*(8/4)     -> 24? True
8*4-10+2         -> 24? True


In [4]:
def count_nodes(trie):
    n, stack = 0, [trie]
    while stack:
        node = stack.pop(); n += 1
        stack.extend(node.children.values())
    return n - 1   # exclude root


class QueryBuffer:
    """Per-query buffer: a PrefixTrie of ALL rollouts ever seen for one query.

    Each `ingest` is one GRPO group (one batch). Scalar advantages are the group
    z-score `a_i = (r_i - mean)/(std + eps)`, inserted token-by-token (here:
    char-level) so shared prefixes accumulate the running max/min advantage.
    """

    def __init__(self, numbers):
        self.numbers = numbers
        self.trie = PrefixTrie()
        self.history = []   # (expr, reward, scalar_adv)

    def ingest(self, completions):
        r = torch.tensor([1.0 if verify_24(self.numbers, c) else 0.0 for c in completions])
        adv = (r - r.mean()) / (r.std(unbiased=False) + 1e-6)
        for c, rew, a in zip(completions, r.tolist(), adv.tolist()):
            self.trie.insert(list(c), a)          # char-level tokens
            self.history.append((c, rew, a))
        return r.tolist(), adv.tolist()


buf = QueryBuffer(NUMBERS)
r0, a0 = buf.ingest(BATCH0)
print("batch0 rewards            :", r0)
print("batch0 scalar advantages  :", [round(x, 3) for x in a0])
print("trie nodes                :", count_nodes(buf.trie))
print("shared-prefix token frac  :", round(buf.trie.shared_prefix_token_fraction, 3))

batch0 rewards            : [1.0, 0.0, 1.0, 0.0]
batch0 scalar advantages  : [1.0, -1.0, 1.0, -1.0]
trie nodes                : 27
shared-prefix token frac  : 0.407


In [5]:
# Second batch on the same query -> the SAME persistent trie grows.
r1, a1 = buf.ingest(BATCH1)
print("batch1 rewards            :", r1)
print("batch1 scalar advantages  :", [round(x, 3) for x in a1])
print("trie nodes (both batches) :", count_nodes(buf.trie))
print("shared-prefix token frac  :", round(buf.trie.shared_prefix_token_fraction, 3))
print("total rollouts buffered   :", len(buf.history))

batch1 rewards            : [0.0, 0.0, 1.0, 1.0]
batch1 scalar advantages  : [-1.0, -1.0, 1.0, 1.0]
trie nodes (both batches) : 48
shared-prefix token frac  : 0.292
total rollouts buffered   : 8


In [6]:
def show_walk(buf, expr, mode="max"):
    toks = list(expr)
    vals = buf.trie.walk(toks, mode=mode)
    print(f"per-token A*({mode}) for {expr!r}:")
    for t, v in zip(toks, vals):
        print(f"   {t!r:>4}  {v:+.3f}")

# "(10-4)*8-2" is WRONG and had a negative advantage within batch1. But its
# prefix "(10-4)*8" is also the prefix of batch0's CORRECT "(10-4)*8/2", so the
# persistent per-query buffer lets the shared prefix inherit that POSITIVE
# (optimistic) advantage -- credit flows from a previous batch's success.
show_walk(buf, "(10-4)*8-2", mode="max")

per-token A*(max) for '(10-4)*8-2':
    '('  +1.000
    '1'  +1.000
    '0'  +1.000
    '-'  +1.000
    '4'  +1.000
    ')'  +1.000
    '*'  +1.000
    '8'  +1.000
    '-'  -1.000
    '2'  -1.000


In [7]:
# Same mechanism via the REAL TreeTrainer credit core. Here `global_tries` IS the
# per-query buffer: a dict (prompt-key -> PrefixTrie) persisted across batches,
# exactly what TreeTrainer keeps in self._global_tries when use_global_tree=True.
PAD = 0
def to_ids(s):  return [ord(c) for c in s]            # toy "tokenizer" (no model needed)
def pad_batch(seqs, pad=PAD):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [pad] * (L - len(s)) for s in seqs])

PROMPT = to_ids("q:" + ",".join(map(str, NUMBERS)))    # identical prompt -> one group/key

def make_inputs(completions):
    cids = pad_batch([to_ids(c) for c in completions])
    mask = (cids != PAD).long()                        # no char maps to 0, so pad-only mask
    pids = pad_batch([PROMPT] * len(completions))
    r = torch.tensor([1.0 if verify_24(NUMBERS, c) else 0.0 for c in completions])
    adv = (r - r.mean()) / (r.std(unbiased=False) + 1e-6)
    return pids, cids, mask, adv

global_tries = {}                                      # <-- the persistent per-query buffer
for name, batch in [("batch0", BATCH0), ("batch1", BATCH1)]:
    pids, cids, mask, adv = make_inputs(batch)
    adv_tok = TreeTrainer._tree_token_advantages(
        pids, cids, mask, adv, PAD,
        use_global_tree=True, global_tries=global_tries, credit_mode="max",
    )
    print(f"{name}: buffered queries = {len(global_tries)} "
          f"(one trie per distinct prompt); adv_tok shape = {tuple(adv_tok.shape)}")

# Per-token credit the trainer would actually pay on a batch1 row, after the
# buffer has seen BOTH batches:
pids, cids, mask, adv = make_inputs(BATCH1)
adv_tok = TreeTrainer._tree_token_advantages(
    pids, cids, mask, adv, PAD,
    use_global_tree=True, global_tries=global_tries, credit_mode="max",
)
row = 0
n = int(mask[row].sum())
print("\nexpr                       :", BATCH1[row])
print("scalar adv (this batch)    :", round(float(adv[row]), 3))
print("per-token A* (from buffer) :", [round(float(x), 3) for x in adv_tok[row][:n]])

batch0: buffered queries = 1 (one trie per distinct prompt); adv_tok shape = (4, 12)
batch1: buffered queries = 1 (one trie per distinct prompt); adv_tok shape = (4, 12)

expr                       : (10-4)*8-2
scalar adv (this batch)    : -1.0
per-token A* (from buffer) : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
